# **Collaborative Filtering based Recommender System using Non-negative Matrix Factorization**


In my previous work, I performed KNN on the user-item interaction matrix to estimate the rating of unknown items based on the aggregation of the user's K nearest neighbors' ratings.


The KNN algorithm is memory-based, which means I need to keep all instances for prediction and maintain a large similarity matrix. This approach can be infeasible if the user/item scale is large. For example, 1 million users would require a 1 million by 1 million similarity matrix, which is very difficult to load into RAM for most computational environments.


#### Non-negative Matrix Factorization


Non-negative Matrix Factorization (NMF) is a dimensionality reduction algorithm that decomposes a large sparse matrix into two smaller and dense matrices.

NMF can be one solution to big matrix issues. The main idea is to decompose the large and sparse user-interaction matrix into two smaller dense matrices: one representing the transformed user features and another representing the transformed item features.


Here's an example: suppose I have a user-item interaction matrix $A$ with 10,000 users and 100 items (10,000 × 100), and its element `(j, k)` represents the rating of item `k` from user `j`. I could decompose $A$ into two smaller and dense matrices $U$ (10,000 × 16) and $I$ (16 × 100). For the user matrix $U$, each row vector is a transformed latent feature vector of a user, and for the item matrix $I$, each column is a transformed latent feature vector of an item.

Here, the dimension 16 is a hyperparameter that defines the size of the hidden user and item features, which means the shape of the transposed user feature vector and item feature vector is now 16 × 1.


When we multiply row `j` of $U$ and column `k` of matrix $I$, we get an estimate of the original rating $\hat{r}_{jk}$.

For example, if I perform the dot product of user one's row vector in $U$ and item one's column vector in $I$, I can get the rating estimation of user one for item one, which is the element (1, 1) in the original interaction matrix $A$.


Note that $I$ is short for Items, and it is not an identity matrix.


We can determine $U$ and $I$ by initializing their components and minimizing the following cost function:


$$\sum_{r_{jk} \in {train}} \left(r_{jk} - \hat{r}_{jk} \right)^2,$$


where $\hat{r}_{jk}$ is the dot product of $u_j^T$ and $i_k$:


$$\hat{r}_{jk} = u_j^Ti_k$$


The cost function can be optimized using stochastic gradient descent (SGD) or other optimization algorithms, similar to training weights in a logistic regression model (with additional steps to ensure the matrices have no negative elements).


## Objectives


In this notebook, I will:


* Perform NMF-based collaborative filtering on the user-item matrix


----


### Loading and Exploring the Dataset


First, let's load our dataset, which is a user-item (learner-course) interaction matrix:


In [1]:
import pandas as pd

In [2]:
rating_df = pd.read_csv("data/ratings.csv")

In [3]:
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3
2,1990814,ML0120ENv3,5
3,380098,BD0211EN,5
4,779563,DS0101EN,3


The dataset contains three columns: `user id`, `item id`, and `rating`. Note that this matrix is presented in dense or vertical form. I can convert it using `pivot` to the original sparse matrix:


In [4]:
rating_sparse_df = rating_df.pivot(index='user', columns='item', values='rating').fillna(0).reset_index().rename_axis(index=None, columns=None)
rating_sparse_df.head()

,user,AI0111EN,BC0101EN,BC0201EN,BC0202EN,BD0101EN,BD0111EN,BD0115EN,BD0121EN,BD0123EN,...,SW0201EN,TA0105,TA0105EN,TA0106EN,TMP0101EN,TMP0105EN,TMP0106,TMP107,WA0101EN,WA0103EN
0,2,0.0,4.0,0.0,0.0,5.0,4.0,0.0,5.0,3.0,...,0.0,5.0,0.0,4.0,0.0,3.0,3.0,0.0,5.0,0.0
1,4,0.0,0.0,0.0,0.0,5.0,3.0,4.0,5.0,3.0,...,0.0,4.0,0.0,0.0,0.0,3.0,3.0,0.0,3.0,3.0
2,5,3.0,5.0,5.0,0.0,4.0,0.0,0.0,0.0,3.0,...,0.0,0.0,4.0,4.0,4.0,4.0,4.0,5.0,0.0,3.0
3,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Next, I need to implement NMF-based collaborative filtering using `Surprise`.


## Implementation with the **Surprise** Library


In [5]:
from surprise import NMF
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

In [6]:
# Save the rating dataframe to a CSV file
rating_df.to_csv("data/course_ratings.csv", index=False)

# Read the course rating dataset with columns user item rating
reader = Reader(line_format='user item rating', sep=',', skip_lines=1, rating_scale=(2, 3))

# Load the dataset from the CSV file
course_dataset = Dataset.load_from_file("course_ratings.csv", reader=reader)

Let's split the data into a training set and a test set:


In [7]:
trainset, testset = train_test_split(course_dataset, test_size=.3)

Now let's check how many users and items we can use to fit the NMF model:


In [8]:
print(f"Total {trainset.n_users} users and {trainset.n_items} items in the training set")

Total 31388 users and 124 items in the training set


## NMF-based Collaborative Filtering


Now I'll fit an NMF model using the training set and evaluate the results using the test set. The code is similar to the KNN-based collaborative filtering, but using the `NMF()` model instead:


In [9]:
# Define the NMF model with some hyperparameters
nmf_model = NMF(verbose=True, random_state=123, n_epochs=50)

# Train the model on the training set
nmf_model.fit(trainset)

# Make predictions on the test set
predictions = nmf_model.test(testset)

# Calculate and display the RMSE
accuracy.rmse(predictions)

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24
Processing epoch 25
Processing epoch 26
Processing epoch 27
Processing epoch 28
Processing epoch 29
Processing epoch 30
Processing epoch 31
Processing epoch 32
Processing epoch 33
Processing epoch 34
Processing epoch 35
Processing epoch 36
Processing epoch 37
Processing epoch 38
Processing epoch 39
Processing epoch 40
Processing epoch 41
Processing epoch 42
Processing epoch 43
Processing epoch 44
Processing epoch 45
Processing epoch 46
Processing epoch 47
Processing epoch 48
Processing epoch 49
RMSE: 1.30

1.3082789275524755

I could try different hyperparameter combinations to see which ones give the best performance. More NMF model hyperparameters can be found in the [Surprise documentation](https://surprise.readthedocs.io/en/stable/matrix_factorization.html).


## Summary


In this notebook, I've implemented NMF-based collaborative filtering. The basic idea is to decompose the original user-item interaction matrix into two smaller and dense user and item matrices. Once I've built these two matrices, I can easily estimate unknown ratings via the dot product of specific rows in the user matrix and specific columns in the item matrix.

This approach offers advantages over KNN-based methods when dealing with large-scale recommendation problems since it doesn't require storing and computing a massive similarity matrix. Instead, it captures latent features in a much more compact form while still providing good rating predictions.